# ASL Translator — MediaPipe Training Notebook

Faster alternative to RTMPose. Uses MediaPipe Hands for keypoint extraction.

| Step | Description | Time |
|---|---|---|
| 1 | Install packages | 2 min |
| 2 | Config + Drive | 1 min |
| 3 | Clone repo | 1 min |
| 4 | Download WLASL videos | 10-20 min |
| 5 | MediaPipe preprocessing | 20-40 min |
| 6 | Train Transformer | 1-2 hours |
| 7 | Evaluate + figures | 5 min |
| 8 | Save to Drive | 1 min |

## 1 — GPU Check

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — go to Runtime → Change runtime type → T4 GPU')

## 2 — Install Packages

In [ ]:
%%capture
import subprocess, sys
packages = [
    'mediapipe',
    'google-generativeai',
    'gdown',
    'scikit-learn',
    'opencv-python-headless',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)
print('All packages installed')

In [ ]:
import mediapipe as mp
import torch, cv2, sklearn
print(f'mediapipe : {mp.__version__}')
print(f'torch     : {torch.__version__}')
print(f'opencv    : {cv2.__version__}')
print('All packages ready')

## 3 — Config & Google Drive

In [ ]:
GITHUB_REPO    = 'https://github.com/YOUR_USERNAME/asl-translator.git'
GEMINI_API_KEY = ''
SAVE_TO_GDRIVE = True

NUM_CLASSES  = 100
BATCH_SIZE   = 64
EPOCHS       = 50
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 10
WINDOW_SIZE  = 32
INPUT_DIM    = 126
D_MODEL      = 256
NHEAD        = 8
NUM_LAYERS   = 4
DIM_FF       = 512
DROPOUT      = 0.3

In [ ]:
import os
if SAVE_TO_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    GDRIVE_DIR = '/content/drive/MyDrive/asl-translator-mp'
    os.makedirs(GDRIVE_DIR, exist_ok=True)
    print(f'Drive mounted: {GDRIVE_DIR}')
else:
    GDRIVE_DIR = None
    print('Not saving to Drive')

## 4 — Clone Repo

In [ ]:
import os
PROJECT_DIR = '/content/asl-translator'

if os.path.exists(PROJECT_DIR):
    print(f'Already exists: {PROJECT_DIR}')
    os.chdir(PROJECT_DIR)
    !git pull origin main
else:
    if 'YOUR_USERNAME' not in GITHUB_REPO:
        !git clone {GITHUB_REPO} {PROJECT_DIR}
        os.chdir(PROJECT_DIR)
    else:
        print('GITHUB_REPO not set — creating folders manually')
        os.makedirs(PROJECT_DIR, exist_ok=True)
        os.chdir(PROJECT_DIR)
        for d in ['data/raw/videos', 'data/keypoints_mp', 'models/checkpoints_mp',
                  'training/logs', 'docs/thesis_figures']:
            os.makedirs(d, exist_ok=True)

print(f'Working dir: {os.getcwd()}')
print(os.listdir('.'))

## 5 — Download WLASL Videos

In [ ]:
import os
os.chdir('/content/asl-translator')

!python data/download_wlasl.py \
    --output-dir data/raw \
    --word-list data/word_list.json \
    --max-per-class 80

In [ ]:
from pathlib import Path

video_dir = Path('data/raw/videos')
if video_dir.exists():
    stats = {d.name: len(list(d.glob('*.mp4'))) for d in sorted(video_dir.iterdir()) if d.is_dir()}
    total = sum(stats.values())
    print(f'Total videos: {total}')
    print(f'Words with videos: {sum(1 for v in stats.values() if v > 0)}/100')
    for word, count in sorted(stats.items(), key=lambda x: -x[1])[:10]:
        print(f'  {word:<20} {count}')
else:
    print('No videos found')

## 6 — MediaPipe Preprocessing

In [ ]:
!python data/preprocess_mediapipe.py \
    --video-dir  data/raw/videos \
    --output-dir data/keypoints_mp

In [ ]:
!python data/preprocess_mediapipe.py --verify --output-dir data/keypoints_mp

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

npy_files = list(Path('data/keypoints_mp').rglob('*.npy'))
if not npy_files:
    print('No .npy files found')
else:
    sample = np.load(str(npy_files[0]))
    print(f'Sample : {npy_files[0].parent.name}/{npy_files[0].name}')
    print(f'Shape  : {sample.shape}  (expected: (32, 126))')
    print(f'Range  : [{sample.min():.3f}, {sample.max():.3f}]')

    right_wrist_x = sample[:, 63]
    right_wrist_y = sample[:, 64]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(right_wrist_x, label='x', color='steelblue')
    axes[0].plot(right_wrist_y, label='y', color='coral')
    axes[0].set(title='Right wrist trajectory', xlabel='Frame', ylabel='Normalised coord')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(right_wrist_x, right_wrist_y, 'o-', color='mediumorchid', alpha=0.7)
    axes[1].set(title='Wrist XY path', xlabel='x', ylabel='y')
    axes[1].invert_yaxis()
    axes[1].grid(alpha=0.3)

    plt.suptitle(f'Word: {npy_files[0].parent.name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('docs/thesis_figures/mp_sample_trajectory.png', dpi=150)
    plt.show()
    print('Saved: docs/thesis_figures/mp_sample_trajectory.png')

## 7 — Model Definition

In [ ]:
import sys
sys.path.insert(0, '/content/asl-translator')

try:
    from models.transformer_mp import ASLTransformerMP, build_model_mp
    print('Loaded ASLTransformerMP from models/transformer_mp.py')
except ImportError:
    print('Defining ASLTransformerMP inline...')
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    class ASLTransformerMP(nn.Module):
        def __init__(self, num_classes=100, input_dim=126, seq_len=32,
                     d_model=256, nhead=8, num_layers=4, dim_feedforward=512, dropout=0.3):
            super().__init__()
            self.input_projection  = nn.Linear(input_dim, d_model)
            self.cls_token         = nn.Parameter(torch.zeros(1, 1, d_model))
            self.pos_embedding     = nn.Parameter(torch.zeros(1, seq_len + 1, d_model))
            nn.init.trunc_normal_(self.cls_token,     std=0.02)
            nn.init.trunc_normal_(self.pos_embedding, std=0.02)
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
                dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
            self.transformer_encoder = nn.TransformerEncoder(
                encoder_layer, num_layers=num_layers, norm=nn.LayerNorm(d_model))
            self.classifier = nn.Sequential(
                nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, num_classes))
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    if m.bias is not None: nn.init.zeros_(m.bias)

        def forward(self, x, padding_mask=None):
            B, T, _ = x.shape
            if padding_mask is None:
                padding_mask = (x.abs().sum(dim=-1) == 0)
            x   = self.input_projection(x)
            cls = self.cls_token.expand(B, -1, -1)
            x   = torch.cat([cls, x], dim=1) + self.pos_embedding
            cls_mask  = torch.zeros(B, 1, dtype=torch.bool, device=x.device)
            full_mask = torch.cat([cls_mask, padding_mask], dim=1)
            x   = self.transformer_encoder(x, src_key_padding_mask=full_mask)
            return self.classifier(x[:, 0, :])

        def count_parameters(self):
            return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def build_model_mp(num_classes=100, device='cpu'):
        m = ASLTransformerMP(num_classes=num_classes).to(device)
        print(f'[model-mp] params: {m.count_parameters():,}')
        return m

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = build_model_mp(num_classes=NUM_CLASSES, device=DEVICE)
dummy  = torch.randn(4, WINDOW_SIZE, INPUT_DIM).to(DEVICE)
out    = model(dummy)
print(f'Input : {dummy.shape}  Output: {out.shape}  (expected [4, {NUM_CLASSES}])')
print(f'Model OK on {DEVICE}')

## 8 — Dataset & DataLoaders

In [ ]:
import json, random
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class WLASLDatasetMP(Dataset):
    def __init__(self, manifest_path, split, keypoints_dir, augment=False):
        self.kd  = Path(keypoints_dir)
        self.aug = augment and split == 'train'
        with open(manifest_path) as f:
            m = json.load(f)
        self.w2i = m['word_to_idx']
        self.i2w = {v: k for k, v in self.w2i.items()}
        raw = m['splits'][split]
        self.samples = [s for s in raw
                        if (self.kd / s['gloss'] / f"{s['video_id']}.npy").exists()]
        miss = len(raw) - len(self.samples)
        if miss: print(f'[{split}] {miss} missing npy files skipped')
        print(f'[{split}] {len(self.samples)} samples | {len(self.w2i)} classes')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s  = self.samples[idx]
        kp = np.load(str(self.kd / s['gloss'] / f"{s['video_id']}.npy"))  # (32, 126)
        if self.aug: kp = self._augment(kp)
        return torch.tensor(kp, dtype=torch.float32), s['label']

    def _augment(self, kp):
        kp = kp.copy()
        if random.random() < 0.5:
            s = random.randint(-3, 3)
            kp = np.roll(kp, s, axis=0)
            if s > 0: kp[:s] = 0
            elif s < 0: kp[s:] = 0
        if random.random() < 0.5:
            n = np.random.normal(0, 0.01, kp.shape).astype(np.float32)
            n[:, 2::3] = 0
            kp += n
        if random.random() < 0.5:
            drop = random.sample(range(32), max(1, int(32 * 0.1)))
            kp[drop] = 0
        if random.random() < 0.5:
            kp[:, 0::3] = -kp[:, 0::3]
        return kp

MANIFEST = 'data/raw/manifest.json'
KP_DIR   = 'data/keypoints_mp'

train_ds = WLASLDatasetMP(MANIFEST, 'train', KP_DIR, augment=True)
val_ds   = WLASLDatasetMP(MANIFEST, 'val',   KP_DIR)
test_ds  = WLASLDatasetMP(MANIFEST, 'test',  KP_DIR)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

xb, yb = next(iter(train_loader))
print(f'Batch x: {xb.shape}  y: {yb.shape}')
print('DataLoaders ready')

## 9 — Training

In [ ]:
import time, json, shutil
from collections import defaultdict
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

def topk_accuracy(logits, labels, topk=(1, 5)):
    with torch.no_grad():
        maxk = max(topk)
        _, pred = logits.topk(maxk, dim=1)
        pred    = pred.t()
        correct = pred.eq(labels.view(1, -1).expand_as(pred))
        return {f'top{k}': correct[:k].reshape(-1).float().mean().item() * 100 for k in topk}

def run_epoch(model, loader, criterion, optimizer=None, device='cuda'):
    training = optimizer is not None
    model.train() if training else model.eval()
    totals = defaultdict(float)
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y   = x.to(device), y.to(device)
            logits = model(x)
            loss   = criterion(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            accs = topk_accuracy(logits, y)
            totals['loss'] += loss.item()
            totals['top1'] += accs['top1']
            totals['top5'] += accs['top5']
    n = len(loader)
    return {k: v / n for k, v in totals.items()}

print('Training helpers ready')

In [ ]:
import os
CKPT_PATH = 'models/checkpoints_mp/best_model_mp.pth'
os.makedirs('models/checkpoints_mp', exist_ok=True)
os.makedirs('training/logs', exist_ok=True)

model     = build_model_mp(num_classes=NUM_CLASSES, device=DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS - 3, eta_min=1e-5)

best_val_top1  = 0.0
patience_count = 0
history        = []

print(f'Training on {DEVICE} | {EPOCHS} epochs | batch={BATCH_SIZE} | lr={LR}')
print('=' * 65)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr = run_epoch(model, train_loader, criterion, optimizer, DEVICE)
    va = run_epoch(model, val_loader,   criterion, None,      DEVICE)
    scheduler.step()

    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]['lr']
    marker  = ''

    if va['top1'] > best_val_top1:
        best_val_top1  = va['top1']
        patience_count = 0
        torch.save({'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'val_top1': best_val_top1}, CKPT_PATH)
        marker = '  <- best'
        if GDRIVE_DIR:
            shutil.copy(CKPT_PATH, f'{GDRIVE_DIR}/best_model_mp.pth')
    else:
        patience_count += 1

    history.append({'epoch': epoch, 'train': tr, 'val': va, 'lr': lr_now})

    print(f'Ep {epoch:3d}/{EPOCHS} | '
          f'tr loss {tr["loss"]:.3f} top1 {tr["top1"]:5.1f}% | '
          f'va loss {va["loss"]:.3f} top1 {va["top1"]:5.1f}% top5 {va["top5"]:5.1f}% | '
          f'lr {lr_now:.2e} | {elapsed:.0f}s{marker}')

    if patience_count >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

with open('training/logs/history_mp.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'Training complete | Best val top-1: {best_val_top1:.1f}%')

## 10 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs  = [h['epoch']         for h in history]
tr_loss = [h['train']['loss'] for h in history]
va_loss = [h['val']['loss']   for h in history]
tr_top1 = [h['train']['top1'] for h in history]
va_top1 = [h['val']['top1']   for h in history]
va_top5 = [h['val']['top5']   for h in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, tr_loss, label='Train')
axes[0].plot(epochs, va_loss, label='Val', linestyle='--')
axes[0].set(title='Loss', xlabel='Epoch', ylabel='Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, tr_top1, label='Train')
axes[1].plot(epochs, va_top1, label='Val', linestyle='--')
axes[1].axhline(max(va_top1), color='green', linestyle=':', label=f'Best: {max(va_top1):.1f}%')
axes[1].set(title='Top-1 Accuracy', xlabel='Epoch', ylabel='Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs, va_top5, color='mediumorchid')
axes[2].axhline(max(va_top5), color='green', linestyle=':', label=f'Best: {max(va_top5):.1f}%')
axes[2].set(title='Val Top-5', xlabel='Epoch', ylabel='Accuracy (%)')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('MediaPipe ASL Transformer — Training', fontsize=13, fontweight='bold')
plt.tight_layout()
out_path = 'docs/thesis_figures/mp_training_curves.png'
plt.savefig(out_path, dpi=150)
plt.show()
print(f'Saved: {out_path}')

## 11 — Evaluation

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded checkpoint (epoch {ckpt["epoch"]}, val top-1: {ckpt["val_top1"]:.1f}%)')

all_true, all_pred, all_conf = [], [], []
top5_correct = 0

with torch.no_grad():
    for x, y in test_loader:
        x      = x.to(DEVICE)
        logits = model(x)
        probs  = torch.softmax(logits, dim=-1)
        preds  = probs.argmax(dim=-1).cpu()
        confs  = probs.max(dim=-1).values.cpu()
        top5   = probs.topk(5, dim=1).indices.cpu()
        for yi, t5 in zip(y, top5):
            if yi.item() in t5.tolist(): top5_correct += 1
        all_true.extend(y.tolist())
        all_pred.extend(preds.tolist())
        all_conf.extend(confs.tolist())

top1_acc = np.mean(np.array(all_true) == np.array(all_pred)) * 100
top5_acc = top5_correct / len(all_true) * 100

with open('data/raw/manifest.json') as f:
    mf = json.load(f)
i2w = {v: k for k, v in mf['word_to_idx'].items()}
class_names = [i2w[i] for i in range(NUM_CLASSES)]

print(f'Test Top-1: {top1_acc:.2f}%')
print(f'Test Top-5: {top5_acc:.2f}%')
print(f'Samples   : {len(all_true)}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

cm         = confusion_matrix(all_true, all_pred)
errors     = cm.sum(axis=1) - np.diag(cm)
top_idx    = np.argsort(errors)[-30:]
cm_sub     = cm[np.ix_(top_idx, top_idx)]
sub_names  = [class_names[i] for i in top_idx]

fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(cm_sub, interpolation='nearest', cmap='Blues')
plt.colorbar(im)
ax.set(xticks=np.arange(30), yticks=np.arange(30),
       xticklabels=sub_names, yticklabels=sub_names,
       xlabel='Predicted', ylabel='True',
       title=f'Confusion Matrix (MediaPipe) — Top-1: {top1_acc:.1f}%  Top-5: {top5_acc:.1f}%')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
thresh = cm_sub.max() / 2
for i in range(30):
    for j in range(30):
        if cm_sub[i,j] > 0:
            ax.text(j, i, str(cm_sub[i,j]), ha='center', va='center', fontsize=6,
                    color='white' if cm_sub[i,j] > thresh else 'black')
plt.tight_layout()
out_path = 'docs/thesis_figures/mp_confusion_matrix.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 12 — Save to Google Drive

In [ ]:
import shutil, json
from pathlib import Path

if GDRIVE_DIR:
    shutil.copy(CKPT_PATH, f'{GDRIVE_DIR}/best_model_mp.pth')

    figs_dst = Path(f'{GDRIVE_DIR}/thesis_figures')
    figs_dst.mkdir(exist_ok=True)
    for f in Path('docs/thesis_figures').glob('mp_*.png'):
        shutil.copy(f, figs_dst / f.name)

    shutil.copy('training/logs/history_mp.json', f'{GDRIVE_DIR}/history_mp.json')

    summary = {
        'model': 'MediaPipe ASLTransformerMP',
        'input_dim': INPUT_DIM,
        'test_top1': top1_acc,
        'test_top5': top5_acc,
        'best_val_top1': best_val_top1,
        'num_classes': NUM_CLASSES,
        'epochs_trained': len(history),
        'params': model.count_parameters(),
    }
    with open(f'{GDRIVE_DIR}/summary_mp.json', 'w') as f:
        json.dump(summary, f, indent=2)

    print(f'Saved to Drive: {GDRIVE_DIR}')
    for p in sorted(Path(GDRIVE_DIR).rglob('*')):
        if p.is_file(): print(f'  {p.relative_to(GDRIVE_DIR)}')
else:
    print(f'Checkpoint at: {CKPT_PATH}')

## Done

| Output | Location |
|---|---|
| Trained model | `models/checkpoints_mp/best_model_mp.pth` |
| Training curves | `docs/thesis_figures/mp_training_curves.png` |
| Confusion matrix | `docs/thesis_figures/mp_confusion_matrix.png` |
| Sample trajectory | `docs/thesis_figures/mp_sample_trajectory.png` |
| Training log | `training/logs/history_mp.json` |

Download `best_model_mp.pth` from Drive and put it in `models/checkpoints_mp/` on your PC.